In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle("C:\\Users\\Udit Anand\\Downloads\\df_train.csv")

In [4]:
print(df.head())

   level_0   index                                          E_channel  \
0     6000    2508  [-0.0, -0.3037471, -0.7941901, -2.0677443, -2....   
1     6001  177771  [0.0, 0.008827989, 0.027690079, 0.050335042, 0...   
2     6002  143069  [-0.0, -0.07114574, -0.07915803, 0.045831945, ...   
3     6003  176271  [-4.039773, -5.0321927, -3.098704, -2.0527992,...   
4     6004   44864  [0.0, 0.0046114475, 0.01731589, 0.057890795, 0...   

                                           N_channel  \
0  [-0.0, -0.1688106, -0.4445446, -1.1667616, -1....   
1  [0.0, 0.0067506083, 0.017026894, 0.0626213, 0....   
2  [-0.0, -0.11352274, -0.40909925, -0.6154829, -...   
3  [-1.1656461, -0.06951267, 2.7692225, 2.9919803...   
4  [-0.0, -0.008839174, -0.03844553, -0.08942565,...   

                                E_channel_precursors  \
0  [-174.67068, -160.9165, -111.053535, -55.44577...   
1  [11.102617, 20.359272, 22.870697, 9.638555, 11...   
2  [550.5055, 621.12805, 494.6476, 227.99667, -8....   


In [5]:
df.columns

Index(['level_0', 'index', 'E_channel', 'N_channel', 'E_channel_precursors',
       'N_channel_precursors', 'Z_channel_precursors', 'Z_channel',
       'back_azimuth_deg', 'coda_end_sample', 'network_code',
       'p_arrival_sample', 'p_status', 'p_travel_sec', 'p_weight',
       'receiver_code', 'receiver_elevation_m', 'receiver_latitude',
       'receiver_longitude', 'receiver_type', 's_arrival_sample', 's_status',
       's_weight', 'snr_db', 'source_depth_km', 'source_depth_uncertainty_km',
       'source_distance_deg', 'source_distance_km', 'source_error_sec',
       'source_gap_deg', 'source_horizontal_uncertainty_km', 'source_id',
       'source_latitude', 'source_longitude', 'source_magnitude',
       'source_magnitude_author', 'source_magnitude_type',
       'source_mechanism_strike_dip_rake', 'source_origin_time',
       'source_origin_uncertainty_sec', 'trace_category', 'trace_name',
       'trace_start_time', 'missing_time_from_event',
       'source_magnitude_label'],
    

In [6]:
df.shape

(34289, 45)

In [7]:
import numpy as np

# convert array column into 2D matrix
signal_matrix = np.vstack(df["Z_channel"].values)

# create new dataframe with expanded columns
signal_df = pd.DataFrame(
    signal_matrix,
    columns=[f"Z_{i}" for i in range(signal_matrix.shape[1])]
)

# drop old array column
df = df.drop(columns=["Z_channel"])

# merge expanded columns
df = pd.concat([df.reset_index(drop=True), signal_df], axis=1)

In [4]:
import pandas as pd
import numpy as np

def split_dataset(df, n_parts=7, rows_per_part=5000, random_state=42):
    """
    Split dataset into n equal parts while:
    - Preserving column dtypes
    - Maintaining statistical distribution (representative samples)
    """
    
    # --- Preserve original dtypes ---
    original_dtypes = df.dtypes.to_dict()
    
    # --- Shuffle while maintaining distribution ---
    df_shuffled = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    parts = []
    for i in range(n_parts):
        start = i * rows_per_part
        end   = start + rows_per_part
        chunk = df_shuffled.iloc[start:end].copy()
        
        # Restore original dtypes (safety check)
        for col, dtype in original_dtypes.items():
            if chunk[col].dtype != dtype:
                chunk[col] = chunk[col].astype(dtype)
        
        chunk = chunk.reset_index(drop=True)
        parts.append(chunk)
    
    return parts


# ─────────────────────────────────────────────
# Usage
# ─────────────────────────────────────────────

parts = split_dataset(df, n_parts=7, rows_per_part=5000)

# Save each part
for i, part in enumerate(parts):
    part.to_csv(f"part_{i+1}.csv", index=False)
    print(f"Part {i+1}: {part.shape} | dtypes match: {dict(part.dtypes) == dict(df.dtypes)}")


# ─────────────────────────────────────────────
# Verify distributions are similar
# ─────────────────────────────────────────────
def verify_distribution(original_df, parts, numeric_cols=None):
    if numeric_cols is None:
        numeric_cols = original_df.select_dtypes(include=np.number).columns.tolist()
    
    print("\n📊 Distribution Comparison (Mean ± Std)")
    print(f"{'Column':<20} {'Original':>20} ", end="")
    for i in range(len(parts)):
        print(f"{'Part '+str(i+1):>15}", end="")
    print()
    print("-" * (20 + 20 + 15 * len(parts)))
    
    for col in numeric_cols[:5]:  # Show first 5 numeric columns
        orig_stat = f"{original_df[col].mean():.2f} ± {original_df[col].std():.2f}"
        print(f"{col:<20} {orig_stat:>20} ", end="")
        for part in parts:
            stat = f"{part[col].mean():.2f}"
            print(f"{stat:>15}", end="")
        print()

verify_distribution(df, parts)

Part 1: (5000, 45) | dtypes match: True
Part 2: (5000, 45) | dtypes match: True
Part 3: (5000, 45) | dtypes match: True
Part 4: (5000, 45) | dtypes match: True
Part 5: (5000, 45) | dtypes match: True
Part 6: (5000, 45) | dtypes match: True
Part 7: (4289, 45) | dtypes match: True

📊 Distribution Comparison (Mean ± Std)
Column                           Original          Part 1         Part 2         Part 3         Part 4         Part 5         Part 6         Part 7
-------------------------------------------------------------------------------------------------------------------------------------------------
level_0                23144.00 ± 9898.53        23132.50       23085.13       23230.11       23168.45       23076.49       22966.72       23382.51
index                 91265.07 ± 61876.36        91397.34       91448.60       91090.54       88831.52       92190.78       92778.54       91093.79
p_arrival_sample           917.27 ± 59.62          917.28         917.94         916.85   

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

def stratified_split(df, target_col, n_parts=7, rows_per_part=5000, random_state=42):
    parts = []
    remaining = df.copy()
    
    for i in range(n_parts):
        # Sample in a stratified manner
        chunk = remaining.groupby(target_col, group_keys=False).apply(
            lambda x: x.sample(
                min(len(x), round(rows_per_part * len(x) / len(remaining))),
                random_state=random_state
            )
        )
        chunk = chunk.sample(min(rows_per_part, len(chunk)), random_state=random_state)
        parts.append(chunk.reset_index(drop=True))
        remaining = remaining.drop(chunk.index)
    
    return parts

# Usage (when you have a target column for ML)
parts = stratified_split(df, target_col="your_label_column")